# CompressAI inference demo

## 1. Inference

### Load Model

In [ ]:
import torch
from model import Net
net = Net()

checkpoint = torch.load("checkpoint_best_loss.pth.tar")
print(f'epoch: {checkpoint["epoch"]}')

net.load_state_dict(checkpoint["state_dict"])
# CPU is sufficient for a single image's Inference
# net.to("cuda")
net.update()

### Load Image

In [ ]:
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("images/airplane.bmp").convert("RGB")
x = transforms.ToTensor()(img).unsqueeze(0)
plt.figure()
plt.axis("off")
plt.imshow(img)
plt.show()

## Packet Loss Simulation

### Encode

In [ ]:
with torch.no_grad():
    tensor = net.g_a(x)
tensor_size = tensor.size()
print(f"tensor size: {tensor_size}")

### Generate Subtensors

In [ ]:
import random
height_indices = list(range(tensor.size(2)))
random.shuffle(height_indices)

chunk_size = 2
height_chunks = [height_indices[i:i + chunk_size] for i in range(0, len(height_indices), chunk_size)]

In [ ]:
subtensors = []
for chunk in height_chunks:
    subtensor = tensor[:, :, chunk, :]
    subtensors.append((chunk, subtensor))

# only the spatial size is needed for decompression
subtensor_size = subtensors[0][1].size()
print(f"subtensor size: {subtensor_size}")
subtensor_spatial_size = subtensor_size[2:]

In [ ]:
packets = []
for chunk, subtensor in subtensors:
    packet = net.entropy_bottleneck.compress(subtensor)
    packets.append((chunk, packet))

### Size Comparison

In [ ]:
tensor_packet = net.entropy_bottleneck.compress(tensor)
print(len(tensor_packet[0]))
print(sum([len(packet[0]) for _, packet in packets]))

### Simulate Packet Loss

In [ ]:
packet_loss_rate = 0.2
rcv_packet_nums = round(len(packets) * (1 - packet_loss_rate))
rcv_packets = packets[:rcv_packet_nums]

print(sum([len(packet[0]) for _, packet in rcv_packets]))

### Decode

In [ ]:
rcv_tensor = torch.zeros(tensor_size)

for chunk, packet in rcv_packets:
    subtensor = net.entropy_bottleneck.decompress(packet, subtensor_spatial_size)
    rcv_tensor[:, :, chunk, :] = subtensor

with torch.no_grad():
    out_net = net.g_s(rcv_tensor)
rec_img = transforms.ToPILImage()(out_net.squeeze())
diff = torch.mean((out_net - x).abs(), axis=1).squeeze()

fix, axes = plt.subplots(1, 3, figsize=(16, 12))
for ax in axes:
    ax.axis("off")

axes[0].imshow(img)
axes[0].title.set_text("Original")

axes[1].imshow(rec_img)
axes[1].title.set_text("Reconstructed")

axes[2].imshow(diff, cmap="viridis")
axes[2].title.set_text("Difference")

plt.show()